# RZ42 Furuta pendulum: parameter identification from encoder data only

This notebook is the procedure for the **real rig**. It uses only what the hardware gives:

| Signal | Source |
|---|---|
| `t` | timestamp of each control-loop iteration (has jitter) |
| `arm_cmd` | arm position command sent to the stepper driver |
| `q1_meas` | stepper (arm) encoder |
| `q2_meas` | AS5600 (pendulum), unwrapped, raw zero |

plus a few bench values taken **on the assembled rig** (`dataset/bench.json`). **No motor torque is used anywhere**: a stepper driver cannot measure it reliably.

In this project the logs come from the hidden MuJoCo plant `mujoco/rz42_real_plant*.xml`. This notebook never opens those files or `dataset/ground_truth.json`; scoring against the truth is done separately in `03_evaluate_against_truth.ipynb`. To run it on the real rig, replace the CSV files with your own logs and update `bench.json`.

## Model used for identification
The stepper is position-controlled, so the measured arm motion $q_1(t)$ is the **input**. Only the pendulum row of the Kane equations (from `01_kane_model.ipynb`) is needed. Dividing it by $J_2$ gives
$$\ddot q_2 = -\kappa\cos q_2\,\ddot q_1 + \rho\,\sin q_2\cos q_2\,\dot q_1^2 - \omega^2\sin q_2 - \beta\,\dot q_2 - \phi\tanh(\dot q_2/v_s)$$

**Identified from the encoders alone** (ratios to $J_2$):

| Ratio | Meaning | Used for |
|---|---|---|
| $\kappa = K/J_2$ | how arm acceleration pushes the pendulum | balance gain, swing-up authority |
| $\rho = J_1/J_2$ | centrifugal effect of arm speed | swing-up and large motions |
| $\omega^2 = G/J_2$ | gravity stiffness ($\sqrt{\omega^2}$ = natural frequency) | balance, swing-up energy |
| $\beta = b_p/J_2$ | 608ZZ viscous damping | model accuracy, friction compensation |
| $\phi = F_p/J_2$ | 608ZZ Coulomb friction | model accuracy, friction compensation |

These five ratios are **all a stepper-based balancing and swing-up controller needs**: it commands the arm acceleration, so it never needs torques or inertias in kg·m².

**Absolute values** ($J_2$, $G$, $K$, $J_1$, $b_p$, $F_p$) need one extra, non-destructive experiment: free decays with a known mass taped to the pendulum (section 5).

**Not identifiable without torque** (section 6): the arm inertia $J_0$, arm friction, and the arm's mass and centre of mass.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.optimize import least_squares

ROOT = Path.cwd().parent if Path.cwd().name == 'identification' else Path.cwd()
DATA = ROOT / 'dataset'
FIG = ROOT / 'identification' / 'figures'
FIG.mkdir(exist_ok=True)
bench = json.loads((DATA / 'bench.json').read_text())
index = json.loads((DATA / 'experiments.json').read_text())['experiments']
g_val = bench['g']
FS = 500.0                     # resampling rate for processing
DT = 1.0 / FS
V_S = 0.01                     # tanh smoothing velocity of the Coulomb term (a modelling choice, rad/s)
for n, v in index.items():
    print(f'{n:26s} {v["purpose"]:11s} {v.get("setup", ""):40s} arm: {v.get("arm", "")}')

## 1. Initial guess: the CAD mass model
The same nominal values as in `01_kane_model.ipynb`. They are used **only** as the starting point of the fit.

In [ ]:
cad = dict(m_p=0.01725, l_p=0.04277, c_p=-0.00177, I_px=1.5613e-5, I_py=1.5204e-5, I_pz=8.93e-7,
           I_pxz=1.309e-6, b_p=1.744e-5, F_p=3.457e-5, r=0.062)
J2_cad = cad['I_px'] + cad['m_p']*cad['l_p']**2
cad_abs = dict(J2=J2_cad, G=g_val*cad['m_p']*cad['l_p'], K=cad['I_pxz'] + cad['m_p']*cad['l_p']*(cad['r'] + cad['c_p']),
               J1=cad['I_py'] - cad['I_pz'] + cad['m_p']*cad['l_p']**2, b_p=cad['b_p'], F_p=cad['F_p'])
RATIOS = ['kappa', 'rho', 'omega2', 'beta', 'phi']
ratios_cad = np.array([cad_abs['K'], cad_abs['J1'], cad_abs['G'], cad_abs['b_p'], cad_abs['F_p']]) / J2_cad
dict(zip(RATIOS, ratios_cad.round(5)))

## 2. Load, resample, calibrate the AS5600 zero, differentiate
1. **Timestamps:** the control loop has jitter, so each log is first interpolated onto a uniform 500 Hz grid using its own timestamps.
2. **AS5600 zero:** the magnet is never mounted at exactly 0. Its offset is the mean of `calib_rest`, recorded with the pendulum hanging still and the arm held. On the real rig: tap the pendulum, let it settle, record 5 s, and repeat a few times. The 608ZZ's Coulomb friction lets it stop a little off-centre.
3. **Derivatives:** a Savitzky–Golay filter (41 samples = 82 ms, order 4) gives velocities and accelerations without phase lag. Half a window is dropped at each end.

In [ ]:
SG_WIN, SG_ORD = 41, 4

def read_log(name):
    raw = np.genfromtxt(DATA / 'experiments' / f'{name}.csv', delimiter=',', names=True)
    return raw['t'], raw['arm_cmd'], raw['q1_meas'], raw['q2_meas']

t_c, _, _, q2_c = read_log('calib_rest')
q2_offset = float(np.mean(q2_c))
print(f'AS5600 zero offset from calib_rest: {np.rad2deg(q2_offset):.3f} deg  (std of the rest signal {np.rad2deg(np.std(q2_c)):.3f} deg)')

dts = np.diff(t_c)
print(f'logged loop period: mean {dts.mean()*1e3:.3f} ms, std {dts.std()*1e6:.0f} us, min {dts.min()*1e3:.3f}, max {dts.max()*1e3:.3f} ms')

def load(name):
    t, cmd, q1, q2 = read_log(name)
    tu = np.arange(t[0], t[-1], DT)
    d = {'t': tu, 'cmd': np.interp(tu, t, cmd), 'q1_raw': np.interp(tu, t, q1),
         'q2_raw': np.interp(tu, t, q2) - q2_offset}
    for j in (1, 2):
        y = d[f'q{j}_raw']
        d[f'q{j}'] = savgol_filter(y, SG_WIN, SG_ORD)
        d[f'qd{j}'] = savgol_filter(y, SG_WIN, SG_ORD, deriv=1, delta=DT)
        d[f'qdd{j}'] = savgol_filter(y, SG_WIN, SG_ORD, deriv=2, delta=DT)
    h = SG_WIN // 2
    return {k: v[h:-h] for k, v in d.items()}

data = {n: load(n) for n in index if n != 'calib_rest'}
train = [n for n, v in index.items() if v['purpose'] == 'train']
scale = [n for n, v in index.items() if v['purpose'] == 'scale']
valid = [n for n, v in index.items() if v['purpose'] == 'validation']
print('train     :', train)
print('scale     :', scale)
print('validation:', valid)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
d = data['sine_A0.15_f1.95']
axes[0].plot(d['t'], np.rad2deg(d['cmd']), color='#999999', lw=1, label='arm command')
axes[0].plot(d['t'], np.rad2deg(d['q1']), color='#2a6fdb', lw=1.2, label='arm q1 (encoder)')
axes[0].set_ylabel('arm [deg]'); axes[0].legend(loc='upper right')
axes[1].plot(d['t'], np.rad2deg(d['q2']), color='#d1495b')
axes[1].set_ylabel('pendulum q2 [deg]')
axes[2].plot(d['t'], d['qdd1'], color='#2a6fdb', lw=1)
axes[2].set_ylabel('arm accel. [rad/s²]'); axes[2].set_xlabel('time [s]')
for a in axes: a.grid(True, alpha=0.3)
axes[0].set_title('Example log: sine_A0.15_f1.95 (near resonance). Arm acceleration is the pendulum\'s input.')
plt.tight_layout(); plt.show()

## 3. Stage A: linear least squares (a quick first estimate)
Move $\ddot q_2$ to the right-hand side; the equation is then linear in the five ratios:
$$\begin{bmatrix} \cos q_2\,\ddot q_1 & -\sin q_2\cos q_2\,\dot q_1^2 & \sin q_2 & \dot q_2 & \tanh(\dot q_2/v_s)\end{bmatrix}\begin{bmatrix}\kappa\\ \rho\\ \omega^2\\ \beta\\ \phi\end{bmatrix} = -\ddot q_2$$
It needs no initial guess, but it relies on the twice-differentiated, noisy $\ddot q_2$, which biases it slightly. Stage B fixes that.

In [ ]:
def regressor(d):
    s, c = np.sin(d['q2']), np.cos(d['q2'])
    return np.column_stack([c*d['qdd1'], -s*c*d['qd1']**2, s, d['qd2'], np.tanh(d['qd2']/V_S)]), -d['qdd2']

Y = np.vstack([regressor(data[n])[0] for n in train])
y = np.concatenate([regressor(data[n])[1] for n in train])
cs = np.abs(Y).max(axis=0)
ratios_A = np.linalg.lstsq(Y / cs, y, rcond=None)[0] / cs
print(f'condition number (scaled regressor): {np.linalg.cond(Y / cs):.1f}')
for nm, a, c0 in zip(RATIOS, ratios_A, ratios_cad):
    print(f'{nm:7s} stage A {a:11.5g}   CAD guess {c0:11.5g}')

## 4. Stage B: output-error fit (the main result)
Simulate the pendulum equation, driven by the **measured** arm motion, and fit the simulated $q_2$ to the measured $q_2$. No derivative of $q_2$ is needed.
* Every training log is cut into 2 s windows. Each window starts from the filtered $q_2,\dot q_2$, so a small model error cannot build up over a whole run.
* All windows are integrated at once (vectorised RK4 at 500 Hz).
* The fit starts from the **CAD guess**, so it shows the procedure works from the prior alone. It is done in log-parameters, which keeps every ratio positive.

In [ ]:
WIN = int(2.0 * FS)

def make_windows(names):
    segs = []
    for n in names:
        d = data[n]
        for k0 in range(0, d['t'].size - WIN, WIN):
            segs.append({k: d[k][k0:k0 + WIN] for k in ('qd1', 'qdd1', 'q2', 'qd2', 'q2_raw')})
    return {k: np.array([s[k] for s in segs]) for k in segs[0]}

def simulate(p, W):
    """RK4 of the pendulum row for every window. p = [kappa, rho, omega2, beta, phi]; W arrays (n_win, N)."""
    kap, rho, w2, beta, phi = p
    qd1, qdd1 = W['qd1'], W['qdd1']
    x = np.array([W['q2'][:, 0], W['qd2'][:, 0]])
    out = np.empty_like(qd1)
    out[:, 0] = x[0]
    def f(x, a, ad):
        s, c = np.sin(x[0]), np.cos(x[0])
        return np.array([x[1], -kap*c*ad + rho*s*c*a**2 - w2*s - beta*x[1] - phi*np.tanh(x[1]/V_S)])
    for k in range(qd1.shape[1] - 1):
        am, bm = 0.5*(qd1[:, k] + qd1[:, k+1]), 0.5*(qdd1[:, k] + qdd1[:, k+1])
        k1 = f(x, qd1[:, k], qdd1[:, k])
        k2 = f(x + 0.5*DT*k1, am, bm)
        k3 = f(x + 0.5*DT*k2, am, bm)
        k4 = f(x + DT*k3, qd1[:, k+1], qdd1[:, k+1])
        x = x + DT*(k1 + 2*k2 + 2*k3 + k4)/6
        out[:, k+1] = x[0]
    return out

def fit_output_error(W, p0, free=(0, 1, 2, 3, 4), fixed=None):
    """least squares on log-parameters; `free` = indices of the ratios that are fitted"""
    p_full = np.array(p0 if fixed is None else fixed, dtype=float)
    def res(lp):
        p = p_full.copy(); p[list(free)] = np.exp(lp)
        return (simulate(p, W) - W['q2_raw']).ravel()
    sol = least_squares(res, np.log(np.asarray(p0, dtype=float)[list(free)]), diff_step=1e-4, max_nfev=200)
    p = p_full.copy(); p[list(free)] = np.exp(sol.x)
    cov = np.linalg.inv(sol.jac.T @ sol.jac) * np.mean(sol.fun**2)       # covariance of the log-parameters
    sd = np.zeros_like(p); sd[list(free)] = p[list(free)] * np.sqrt(np.diag(cov))
    return p, sd, sol

W_train = make_windows(train)
rms0 = np.sqrt(np.mean((simulate(ratios_cad, W_train) - W_train['q2_raw'])**2))
ratios_B, sd_B, solB = fit_output_error(W_train, ratios_cad)
print(f'{W_train["q2"].shape[0]} windows of {WIN} samples; {solB.nfev} model evaluations')
print(f'q2 RMS error: CAD guess {np.rad2deg(rms0):.2f} deg -> identified {np.rad2deg(np.sqrt(np.mean(solB.fun**2))):.3f} deg '
      f'(AS5600 resolution {360/4096:.3f} deg)')
for nm, a, sd, b in zip(RATIOS, ratios_B, sd_B, ratios_A):
    print(f'{nm:7s} = {a:11.5g} +- {sd:8.2g}   (stage A {b:.5g})')

### Repeatability check
The ±1σ values above come from the fit's Jacobian. They assume white, independent errors, so they are optimistic.
A more honest check, exactly as you would do with real data: split the experiments into two groups, fit each group on its own, and compare.
* group 1: free decays + sines;
* group 2: free decays + chirp, swings and pumping.

The free decays are in both groups, because they are the only experiments that isolate the friction.

In [ ]:
decays = [n for n in train if n.startswith('free_decay')]
g1 = decays + [n for n in train if n.startswith('sine')]
g2 = decays + [n for n in train if not n.startswith('sine') and not n.startswith('free_decay')]
rep = {}
for lab, grp in [('group 1', g1), ('group 2', g2)]:
    rep[lab] = fit_output_error(make_windows(grp), ratios_cad)[0]
print(f'{"ratio":7s} {"all data":>11s} {"group 1":>11s} {"group 2":>11s}  spread')
for i, nm in enumerate(RATIOS):
    a, b1, b2 = ratios_B[i], rep['group 1'][i], rep['group 2'][i]
    print(f'{nm:7s} {a:11.5g} {b1:11.5g} {b2:11.5g}  {100*abs(b1 - b2)/a:5.2f} %')

## 5. Absolute scale: the added-mass experiment (rig stays assembled)
The ratios fix everything up to one unknown scale, $J_2$. To get it without torque, tape a known mass $\Delta m$ to the pendulum centre line, at a measured distance $d$ below the pivot, and repeat the free decays (`massD1_*`).
This adds a known gravity moment and a known inertia:
$$\omega_1^2 = \frac{G + g\,\Delta m\, d}{J_2 + \Delta m\, d^2 + I_{\Delta}}, \qquad G = \omega^2 J_2
\quad\Rightarrow\quad J_2 = \frac{g\,\Delta m\, d - \omega_1^2(\Delta m\,d^2 + I_\Delta)}{\omega_1^2 - \omega^2}$$
where $I_\Delta = \Delta m\,r_\Delta^2/2$ is the mass's own inertia about the hinge direction, from its measured size.

**Design rule.** Put the mass well away from $d^* = g/\omega^2 \approx 66$ mm. At $d^*$ the swing frequency does not change at all and $J_2$ cannot be resolved. With $d_1 = 25$ mm and 10 g, $\omega^2$ rises by about 9 %.

Then $\omega_1^2$ (and the friction ratios with the mass) are fitted on the added-mass decays. $\kappa$ and $\rho$ are fixed at their Stage B values; they hardly matter, because the arm is held.

In [ ]:
dm, d1 = bench['added_mass_kg'], bench['added_mass_d1_m']
r_add = bench['added_mass_dimensions_m']['diameter'] / 2
I_add = dm * r_add**2 / 2

W_m1 = make_windows(scale)
p_m1, sd_m1, _ = fit_output_error(W_m1, ratios_B, free=(2, 3, 4), fixed=ratios_B)
w2, w2_1 = ratios_B[2], p_m1[2]
print(f'omega^2 without mass {w2:.4f}, with mass at d1 {w2_1:.4f}  ({100*(w2_1/w2 - 1):+.2f} %)')

def J2_from(dm, d, w2, w2_1, I_add):
    return (g_val*dm*d - w2_1*(dm*d**2 + I_add)) / (w2_1 - w2)

J2_abs = J2_from(dm, d1, w2, w2_1, I_add)

# uncertainty: Monte Carlo over the bench measurement errors (scale 0.05 g rms, calipers 0.3 mm rms, size 10 %)
rng = np.random.default_rng(0)
mc = J2_from(dm + rng.normal(0, 5e-5, 20000), d1 + rng.normal(0, 3e-4, 20000),
             w2 + rng.normal(0, sd_B[2], 20000), w2_1 + rng.normal(0, sd_m1[2], 20000),
             I_add * (1 + rng.normal(0, 0.1, 20000)))
J2_sd = mc.std()
print(f'J2 = {J2_abs:.4e} kg m^2  +- {J2_sd:.1e} ({100*J2_sd/J2_abs:.1f} %, dominated by the caliper measurement of d)')

abs_id = dict(J2=J2_abs, G=ratios_B[2]*J2_abs, K=ratios_B[0]*J2_abs, J1=ratios_B[1]*J2_abs,
              b_p=ratios_B[3]*J2_abs, F_p=ratios_B[4]*J2_abs)
abs_sd = {k: abs(v)*np.hypot(J2_sd/J2_abs, (sd_B[i]/ratios_B[i]) if i is not None else 0)
          for (k, v), i in zip(abs_id.items(), [None, 2, 0, 1, 3, 4])}
for k in abs_id:
    print(f'{k:4s} = {abs_id[k]:11.4e} +- {abs_sd[k]:8.1e}   (CAD guess {cad_abs[k]:.4e})')

### Pendulum centre of mass (optional)
$G = g\,m_p l_p$. If the pendulum was weighed **before assembly**, $l_p = G/(g\,m_p)$ follows. Otherwise only the gravity moment $G$ is known, and that is enough for control.
The hinge-direction offset only appears in the combination $c_p + I_{pxz}/(m_p l_p) = K/(m_p l_p) - r$. The two parts cannot be separated with these measurements.

In [ ]:
if bench.get('m_pendulum_kg'):
    m_p = bench['m_pendulum_kg']
    l_p = abs_id['G'] / (g_val * m_p)
    l_p_sd = abs_sd['G'] / (g_val * m_p)
    cp_comb = abs_id['K'] / (m_p * l_p) - bench['r_m']
    I_px = abs_id['J2'] - m_p * l_p**2
    print(f'pendulum COM below the pivot  l_p = {l_p*1e3:.2f} +- {l_p_sd*1e3:.2f} mm   (CAD {cad["l_p"]*1e3:.2f} mm)')
    print(f'hinge-offset combination c_p + I_pxz/(m_p l_p) = {cp_comb*1e3:+.2f} mm   (CAD {(cad["c_p"] + cad["I_pxz"]/(cad["m_p"]*cad["l_p"]))*1e3:+.2f} mm)')
    print(f'pendulum inertia about its COM, hinge axis  I_px = {I_px:.3e} kg m^2   (CAD {cad["I_px"]:.3e})')
else:
    m_p = l_p = l_p_sd = cp_comb = I_px = None
    print('m_p not given: centre of mass not reported (G = g m_p l_p is known)')

## 6. What these measurements cannot identify
| Quantity | Why not | Does it matter? |
|---|---|---|
| arm inertia $J_0$ (arm + rotor + pendulum about the motor axis) | It only appears in the arm row, where the unknown motor torque is. With a position-controlled stepper, the arm motion is imposed whatever $J_0$ is. | Not for the controller. It matters for the stepper's torque margin and stall risk; use the CAD estimate with a safety factor. |
| arm friction $b_a$, $F_a$ | Same reason: arm row only. | Not for the controller (the driver overcomes it). |
| arm mass $m_a$ and centre of mass $x_a$ | The arm turns about a vertical axis, so gravity does no work on it; they only appear inside $J_0$. | No. |
| $m_p$ and $l_p$ separately | Only the product $G = g\,m_p l_p$ appears. | No. $G$ is what matters; $l_p$ needs a pre-assembly weighing (section 5). |
| $I_{py}$, $I_{pz}$ separately | Only $J_1 \propto I_{py} - I_{pz} + m_p l_p^2$ appears. | No. |
| $c_p$ and $I_{pxz}$ separately | Only $K = I_{pxz} + m_p l_p(r + c_p)$ appears. | No. |
| the true Coulomb law at very low speed | $\tanh$ with $v_s$ = 0.01 rad/s is a model choice. | Slightly, for the last degrees of a decay. |

If you ever need $J_0$: the clean way is a torque-controlled motor, or a driver with a calibrated FOC current reading. A stepper's position error or phase current is not a reliable torque sensor.

## 7. Validation on independent experiments
All four validation runs can be repeated on the real rig, and none was used in the fit:
* `multisine_val`: arm multisine;
* `swing_A0.60_f1.30_val`: large arm swing;
* `free_decay_045deg_val`: a free decay from a release angle not used in training;
* `massD2_decay_040deg_val`: added mass at a **different** distance $d_2$. This is a **prediction of the absolute scale**: the frequency and decay with the mass at $d_2$ are predicted from $J_2$, $G$ and the friction, without refitting anything.

Each run is simulated **from start to end without restarts**, driven by its measured arm motion. The score is the NRMSE fit, $100\,(1 - \lVert q_2 - \hat q_2\rVert/\lVert q_2 - \bar q_2\rVert)$ %.

In [ ]:
def fit_pct(yv, yhat):
    return 100 * (1 - np.linalg.norm(yv - yhat) / np.linalg.norm(yv - yv.mean()))

def ratios_with_mass(J2, G, K, J1, b_p, F_p, dm, d, I_add, r):
    """ratios of the pendulum carrying a point-like added mass at (0, 0, -d) in the pendulum frame"""
    J2m, Gm = J2 + dm*d**2 + I_add, G + g_val*dm*d
    Km, J1m = K + dm*d*r, J1 + dm*d**2
    return np.array([Km, J1m, Gm, b_p, F_p]) / J2m

d2 = bench['added_mass_d2_m']
pred = {}
for n in valid:
    if n.startswith('massD2'):
        pred[n] = {'CAD guess': ratios_with_mass(*[cad_abs[k] for k in ('J2', 'G', 'K', 'J1', 'b_p', 'F_p')], dm, d2, I_add, bench['r_m']),
                   'identified': ratios_with_mass(*[abs_id[k] for k in ('J2', 'G', 'K', 'J1', 'b_p', 'F_p')], dm, d2, I_add, bench['r_m'])}
    else:
        pred[n] = {'CAD guess': ratios_cad, 'identified': ratios_B}

scores = {}
fig, axes = plt.subplots(len(valid), 1, figsize=(10, 3.2*len(valid)))
for ax, n in zip(axes, valid):
    d = data[n]
    W = {k: d[k][None, :] for k in ('qd1', 'qdd1', 'q2', 'qd2')}
    ax.plot(d['t'], np.rad2deg(d['q2_raw']), color='#333333', lw=1.0, label='measured')
    for lab, col, ls in [('CAD guess', '#e09f3e', '--'), ('identified', '#2a6fdb', '-')]:
        q2_hat = simulate(pred[n][lab], W)[0]
        scores[(n, lab)] = fit_pct(d['q2_raw'], q2_hat)
        ax.plot(d['t'], np.rad2deg(q2_hat), ls, color=col, lw=1.1, label=f'{lab} (fit {scores[(n, lab)]:.1f} %)')
    ax.set_title(f'validation: {n}' + ('   [absolute scale predicted for d2, nothing refitted]' if n.startswith('massD2') else ''))
    ax.set_ylabel('q2 [deg]'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('time [s]')
plt.tight_layout(); fig.savefig(FIG / 'validation.png', dpi=110); plt.show()
for (n, lab), s in scores.items():
    print(f'{n:26s} {lab:11s} fit {s:6.1f} %')

In [ ]:
# frequency check of the absolute scale: measured omega^2 with the mass at d2 vs. predicted
W_m2 = make_windows([n for n in valid if n.startswith('massD2')])
p_m2_meas = fit_output_error(W_m2, ratios_B, free=(2,), fixed=pred['massD2_decay_040deg_val']['identified'])[0]
w2_2_pred = pred['massD2_decay_040deg_val']['identified'][2]
print(f'mass at d2: measured omega^2 {p_m2_meas[2]:.4f}, predicted {w2_2_pred:.4f}  '
      f'({100*(w2_2_pred/p_m2_meas[2] - 1):+.2f} %); without the mass {w2:.4f}')

## 8. Model for balancing and swing-up control
**Balancing.** The controller commands the arm acceleration $a = \ddot q_1$; the stepper driver turns it into steps. Linearising at upright ($q_2 = \pi + \delta$), with state $x = [q_1, \delta, \dot q_1, \dot\delta]$:
$$\dot x = \begin{bmatrix}0&0&1&0\\0&0&0&1\\0&0&0&0\\0&\omega^2&0&-\beta_{\mathrm{lin}}\end{bmatrix}x + \begin{bmatrix}0\\0\\1\\ \kappa\end{bmatrix}a$$
Here $\cos\pi = -1$, which turns $-\kappa\cos q_2$ into $+\kappa$.

**Swing-up.** Energy per unit $J_2$, measured from hanging:
$$\tilde E = \tfrac12\dot q_2^2 + \omega^2(1 - \cos q_2), \qquad \tilde E_{\mathrm{upright}} = 2\omega^2$$
An energy-pumping law (e.g. $a = k\,(\tilde E_{\mathrm{up}} - \tilde E)\,\mathrm{sign}(\dot q_2\cos q_2)$) needs only $\omega^2$ and $\kappa$; $\beta$ and $\phi$ give the friction loss per swing to compensate.

In [ ]:
beta_lin = ratios_B[3]   # viscous part only: the tanh slope F/v_s at zero speed is a smoothing artefact
A_up = np.array([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 0], [0, ratios_B[2], 0, -beta_lin]])
B_up = np.array([[0], [0], [1], [ratios_B[0]]])
ctrb = np.hstack([np.linalg.matrix_power(A_up, k) @ B_up for k in range(4)])
print('A_up =\n', A_up.round(4)); print('B_up =\n', B_up.round(4))
print('controllability rank:', np.linalg.matrix_rank(ctrb), '  open-loop eigenvalues:', np.linalg.eigvals(A_up).round(3))
print(f'natural frequency hanging (arm held): {np.sqrt(ratios_B[2])/(2*np.pi):.4f} Hz;  unstable pole at upright: +{np.sqrt(ratios_B[2]):.3f} 1/s')
print(f'swing-up energy target E_up/J2 = 2 omega^2 = {2*ratios_B[2]:.2f} rad^2/s^2')

## 9. Save the identified model
`identified_parameters.json` is the hand-over to the controller and to `03_evaluate_against_truth.ipynb`. Nothing in it came from the hidden plant.

In [ ]:
out = {
    'procedure': 'encoder-only (q1, q2, timestamps, arm command); no motor torque',
    'model': 'qdd2 = -kappa cos(q2) qdd1 + rho sin(q2)cos(q2) qd1^2 - omega2 sin(q2) - beta qd2 - phi tanh(qd2/v_s); q2 = 0 hanging',
    'v_s': V_S, 'savgol': [SG_WIN, SG_ORD], 'as5600_zero_offset_rad': q2_offset,
    'ratios': dict(zip(RATIOS, ratios_B)), 'ratios_1sigma': dict(zip(RATIOS, sd_B)),
    'ratios_stageA': dict(zip(RATIOS, ratios_A)),
    'repeatability': {k: dict(zip(RATIOS, v)) for k, v in rep.items()},
    'absolute_from_added_mass': abs_id, 'absolute_1sigma': abs_sd,
    'pendulum_com': None if l_p is None else {'l_p': l_p, 'l_p_1sigma': l_p_sd, 'c_p_plus_Ipxz_over_mplp': cp_comb, 'I_px': I_px},
    'not_identifiable': ['J0 (arm inertia about motor axis)', 'b_a, F_a (arm friction)', 'm_a, x_a (arm mass, COM)',
                         'm_p and l_p separately (only G = g m_p l_p; l_p only with a pre-assembly weighing)',
                         'I_py, I_pz separately', 'c_p and I_pxz separately'],
    'control_model_upright': {'A': A_up.tolist(), 'B': B_up.tolist(), 'state': ['q1', 'q2 - pi', 'qd1', 'qd2'], 'input': 'arm acceleration'},
    'validation_fit_pct': {f'{n} | {lab}': s for (n, lab), s in scores.items()},
}
(ROOT / 'identification' / 'identified_parameters.json').write_text(json.dumps(out, indent=2, default=float))
print('wrote identified_parameters.json')